# GNR653 HW0: environment check
Run every cell from top to bottom using Shift+Enter or click **Run All** from the menu. The last cell prints a summary.
**TASK**: Take a screenshot of the summary and upload it to Moodle.

- In a codespace, choose the kernel **GNR653 (uv)** or the `.venv` interpreter (top right of the notebook).

## 1. Python and the environment

In [ ]:
import platform, os, sys
from pathlib import Path

results = {}  # check name -> (passed, detail)

def record(name, passed, detail=""):
    results[name] = (passed, detail)
    print(("PASS " if passed else "FAIL ") + name + (f": {detail}" if detail else ""))

py_ok = sys.version_info[:2] == (3, 12)
record("Python version", py_ok, platform.python_version())

in_venv = ".venv" in sys.executable
record("Using the course .venv", in_venv, sys.executable)

## 2. Course packages

In [ ]:
import importlib

PACKAGES = [
    "numpy", "pandas", "matplotlib", "scipy",
    "geopandas", "pyogrio", "shapely", "pyproj", "folium",
    "rasterio", "rioxarray", "rasterstats",
    "xarray", "netCDF4", "zarr", "dask", "distributed",
    "pystac_client", "odc.stac", "gstools",
]

missing = []
for name in PACKAGES:
    try:
        mod = importlib.import_module(name)
        print(f"  {name:<14} {getattr(mod, '__version__', 'ok')}")
    except Exception as e:
        missing.append(name)
        print(f"  {name:<14} MISSING ({type(e).__name__})")

record("All course packages import", not missing, ", ".join(missing) if missing else f"{len(PACKAGES)} packages")

import rasterio
record("GDAL inside rasterio", True, f"GDAL {rasterio.__gdal_version__}")

## 3. Command-line tools

In [ ]:
import shutil, subprocess

def tool_version(cmd):
    exe = shutil.which(cmd[0])
    if exe is None:
        return None
    out = subprocess.run(cmd, capture_output=True, text=True)
    return (out.stdout or out.stderr).strip().splitlines()[0]

tools = {
    "git": ["git", "--version"],
    "rio (from rasterio, via uv)": ["rio", "--version"],
    "gdalinfo (GDAL CLI, via pixi)": ["gdalinfo", "--version"],
}
tools["uv"] = ["uv", "--version"]
tools["pixi"] = ["pixi", "--version"]

# rio lives in the .venv; make sure it is on PATH for this check
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

for label, cmd in tools.items():
    v = tool_version(cmd)
    record(label, v is not None, v or "not found on PATH")

## 4. A small offline geospatial computation

In [ ]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import xarray as xr
import rioxarray  # noqa: F401  (adds the .rio accessor)

# Vector: IIT Bombay main gate, buffered by 1 km in UTM zone 43N
gate = gpd.GeoDataFrame(geometry=[Point(72.9133, 19.1334)], crs="EPSG:4326")
buf = gate.to_crs("EPSG:32643").buffer(1000)
area_km2 = float(buf.area.iloc[0]) / 1e6

# Raster: a small synthetic grid in WGS84, reprojected to UTM 43N
lat = np.linspace(19.20, 19.05, 60)
lon = np.linspace(72.85, 73.00, 60)
da = xr.DataArray(np.random.default_rng(0).random((60, 60)), coords={"y": lat, "x": lon}, dims=("y", "x"))
da = da.rio.write_crs("EPSG:4326")
utm = da.rio.reproject("EPSG:32643")

ok = abs(area_km2 - np.pi) < 0.01 and utm.rio.crs.to_epsg() == 32643
record("Vector + raster operations", ok, f"buffer area {area_km2:.3f} km2, raster reprojected to EPSG:{utm.rio.crs.to_epsg()}")

## 5. Reading satellite data from the cloud (STAC + COG)
This needs internet access. It searches for a recent Sentinel-2 scene over Powai and reads a small window.

In [ ]:
try:
    from pystac_client import Client
    import rasterio
    from rasterio.windows import Window

    catalog = Client.open("https://earth-search.aws.element84.com/v1")
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        intersects={"type": "Point", "coordinates": [72.9133, 19.1334]},
        datetime="2025-11-01/2026-03-31",
        query={"eo:cloud_cover": {"lt": 10}},
        max_items=1,
    )
    item = next(search.items())
    href = item.assets["red"].href
    with rasterio.Env(AWS_NO_SIGN_REQUEST="YES"):
        with rasterio.open(href) as src:
            block = src.read(1, window=Window(0, 0, 256, 256))
    record("STAC search + COG window read", True, f"{item.id}, window {block.shape}, mean {block.mean():.0f}")
except Exception as e:
    record("STAC search + COG window read", False, f"{type(e).__name__}: {e}")

## 6. Summary (screenshot this cell's output for Moodle)

In [ ]:
from datetime import datetime, timezone, timedelta

ist = timezone(timedelta(hours=5, minutes=30))
n_pass = sum(p for p, _ in results.values())
print("=" * 60)
print("GNR653 HW0 environment check")
print("Time (IST):   ", datetime.now(ist).strftime("%Y-%m-%d %H:%M"))
print("GitHub user:  ", os.environ.get("GITHUB_USER", "(not in a codespace)"))
print("Codespace:    ", os.environ.get("CODESPACE_NAME", "(not in a codespace)"))
print("Platform:     ", platform.platform())
print("-" * 60)
for name, (passed, detail) in results.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")
print("-" * 60)
print(f"{n_pass}/{len(results)} checks passed")
print("ALL CHECKS PASSED" if n_pass == len(results) else "Some checks failed: see the FAIL lines above and the HW0 troubleshooting notes.")
print("=" * 60)